In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

In [2]:
#Read PDF from folder
loader = PyPDFDirectoryLoader("./us_census")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
final_docs = text_splitter.split_documents(documents)

In [3]:
# Embedding using HuggingFace
huggingface_embeddings = HuggingFaceBgeEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device':'cpu'},
    encode_kwargs = {'normalize_embeddings':True}
)

c:\Users\reddy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Users\reddy\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
#For testing
import  numpy as np
print(np.array(huggingface_embeddings.embed_query(final_docs[0].page_content)))
print(np.array(huggingface_embeddings.embed_query(final_docs[0].page_content)).shape)

[-2.10483130e-02  4.11389163e-03  4.37868610e-02  2.05080844e-02
  8.60858485e-02  1.18468672e-01 -1.80744324e-02 -1.42715042e-02
 -1.06268682e-01  1.45182572e-02  5.15961275e-03  1.24929667e-01
 -2.23489478e-02 -7.79558569e-02  3.47871669e-02  4.62152297e-03
  1.10306107e-02 -4.85650599e-02 -1.33699710e-02  5.70637099e-02
 -2.73552630e-02  4.26027253e-02 -3.15802433e-02 -4.72046249e-03
  3.28680389e-02  9.39553231e-03  4.56156656e-02  2.50200219e-02
  6.39714627e-03  5.87833337e-02  7.38392919e-02  6.23270217e-03
  3.24792636e-04 -8.62447638e-03  3.61743607e-02 -4.14577387e-02
 -3.59882973e-02  1.16341012e-02 -9.20130834e-02  1.53853912e-02
 -2.69169845e-02 -6.63942471e-02 -9.50659513e-02  8.23578984e-02
  3.13828252e-02  1.11855939e-02 -5.28727099e-02  8.54620934e-02
 -1.69520173e-02  4.13136967e-02  6.34124409e-03  3.36876884e-02
  3.86929326e-02 -1.74281411e-02  2.11679153e-02 -3.26560512e-02
 -2.61763241e-02 -2.57137995e-02 -3.13347764e-02  6.01034553e-04
 -1.27449222e-02 -2.60376

In [5]:
## VectorStore Creation
vectorstore = FAISS.from_documents(final_docs[:120],huggingface_embeddings)

In [6]:
#Query using Similarity Search
query = "WHAT IS THE AMERICAN COMMUNITY SURVEY?"
relevant_docs = vectorstore.similarity_search(query)

print(relevant_docs[0].page_content)

The American Community Survey (ACS) is a continuous survey, and 
people respond throughout the year. Since income is reported for the previous 12 months, the appropriate poverty threshold for each family is determined by multiplying the base-year poverty threshold from 1982 by the average of monthly CPI-U values for the 12 months preceding the survey month. 
For more information, refer to page 110 of “American Community 
Survey and Puerto Rico Community Survey 2022 Subject Definitions” at <www.census.gov/programs-surveys/acs/technical-documentation/
code-lists.html>. For more information on ACS sample design and other topics, refer to <www.census.gov/acs>.


In [7]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000243D4B36240> search_kwargs={'k': 3}


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()

login(token = os.environ["HUGGINGFACEHUB_API_TOKEN"])

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to C:\Users\reddy\.cache\huggingface\token
Login successful


In [10]:
from langchain_community.llms import HuggingFaceHub

hf = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-v0.3",
    model_kwargs={"temperature":0.1,"max_length":500}
)
query="WHAT IS THE AMERICAN COMMUNITY SURVEY?"
hf.invoke(query)

c:\Users\reddy\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEndpoint`.
  warn_deprecated(


HfHubHTTPError:  (Request ID: phH_svcQ-aNQO_VrdHRZH)

403 Forbidden: None.
Cannot access content at: https://api-inference.huggingface.co/models/mistralai/Mistral-7B-v0.3.
If you are trying to create or update content,make sure you have a token with the `write` role.
The model mistralai/Mistral-7B-v0.3 is too large to be loaded automatically (14GB > 10GB). Please use Spaces (https://huggingface.co/spaces) or Inference Endpoints (https://huggingface.co/inference-endpoints).

In [ ]:
#Hugging Face models can be run locally through the HuggingFacePipeline class.
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.3",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf 
llm.invoke(query)

ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.